<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/alpaca_llm_judge_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
import json
from pathlib import Path

FILE = Path(
"/content/drive/MyDrive/openend2/alpaca/raw_inputs/arabic_esl_paper.jsonl"
)

with open(FILE,"r",encoding="utf-8") as f:
    row = json.loads(next(f))

print(row.keys())

for k,v in row.items():

    if isinstance(v,str):

        print("\n================")
        print(k)
        print(v[:500])

dict_keys(['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'text_transformed', 'applied_rules'])

datasplit
eval

dataset
helpful_base

instruction
What are the names of some famous actors that started their careers on Broadway?

input


output
Some famous actors that started their careers on Broadway include Nathan Lane, Audra McDonald, Sutton Foster, and Idina Menzel.

generator
text-davinci-003

sample_mode
temp=0.7,top_p=1.0,max_new_tokens=300

text_transformed
What are the names of some famous actors, each of whom started their career on Broadway?


In [42]:
import json
import random
import pandas as pd
from pathlib import Path

# ============================================================
# Settings
# ============================================================

ROOT = Path("/content/drive/MyDrive/openend2")
INPUT_ROOT = ROOT / "alpaca"

EXPORT_DIR = ROOT / "alpaca_llm_judge_validation"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
display_rng = random.Random(RANDOM_SEED + 1000)

CONDITION_KEYS = [
    "clean",
    "esl_only",
    "typo_only",
    "combined",
]

CONDITION_LABELS = {
    "clean": "Clean",
    "esl_only": "ESL-only",
    "typo_only": "Typo-only",
    "combined": "Combined",
}

BASELINE_KEYS = [
    "reference_output",
    "baseline_output",
    "output_ref",
    "chosen",
    "reference",
    "output",
]


# ============================================================
# Helper functions
# ============================================================

def read_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(
                    f"Warning: invalid JSON in {path}, "
                    f"line {line_number}"
                )

    return rows


input_file_cache = {}


def load_source_rows(source_file):
    """
    Examples:
    raw_inputs/test.jsonl
    raw_inputs/arabic_esl_paper.jsonl
    typo_transforms/rate_0.4/test.jsonl
    typo_transforms/rate_0.4/arabic_esl_paper.jsonl
    """

    source_file = str(source_file).replace("\\", "/")
    source_path = INPUT_ROOT / source_file

    if source_path not in input_file_cache:
        if not source_path.exists():
            raise FileNotFoundError(
                f"Input source file not found: {source_path}"
            )

        input_file_cache[source_path] = read_jsonl(source_path)

    return input_file_cache[source_path]


def get_base_row_index(base_id):
    """
    row-688 -> 688
    """

    text = str(base_id)

    if not text.startswith("row-"):
        raise ValueError(f"Unexpected base_id: {base_id}")

    return int(text.split("-")[1])


def extract_baseline_response(source_row):
    """
    In your AlpacaFarm input files, 'output' is the
    text-davinci-003 baseline response.
    """

    for key in BASELINE_KEYS:
        value = source_row.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip(), key

    return "", None


def normalize_llm_result(model_pref):
    """
    From run_alpaca_farm.py:
    2 = evaluated model wins
    1 = baseline wins
    0 = tie
    """

    mapping = {
        2: "MODEL",
        1: "BASELINE",
        0: "TIE",
    }

    return mapping.get(model_pref, "UNKNOWN")


# ============================================================
# Build 96 comparisons
# ============================================================

review_rows = []
answer_key_rows = []

evaluation_number = 1

for unit in sampled_units:
    base_index = get_base_row_index(unit["base_id"])

    for condition_key in CONDITION_KEYS:
        condition_data = unit[condition_key]

        output_row = condition_data["output_row"]
        score_row = condition_data["score_row"]

        source_file = (
            output_row.get("source_file")
            or score_row.get("source_file")
        )

        source_rows = load_source_rows(source_file)

        if base_index >= len(source_rows):
            raise IndexError(
                f"Row index {base_index} is outside {source_file}; "
                f"file contains {len(source_rows)} rows."
            )

        source_row = source_rows[base_index]

        baseline_response, baseline_field = (
            extract_baseline_response(source_row)
        )

        model_response = str(
            output_row.get("output", "")
        ).strip()

        instruction = str(
            output_row.get(
                "instruction",
                source_row.get("instruction", "")
            )
        ).strip()

        additional_input = str(
            output_row.get(
                "input",
                source_row.get("input", "")
            )
        ).strip()

        if additional_input:
            displayed_prompt = (
                f"{instruction}\n\n"
                f"Additional input:\n{additional_input}"
            )
        else:
            displayed_prompt = instruction

        if not baseline_response:
            raise ValueError(
                f"Missing baseline response for "
                f"{unit['matched_unit_id']} / {condition_key}"
            )

        if not model_response:
            raise ValueError(
                f"Missing model response for "
                f"{unit['matched_unit_id']} / {condition_key}"
            )

        # Randomize model/baseline placement for human reviewers.
        displayed_model_is_a = display_rng.choice([True, False])

        if displayed_model_is_a:
            response_a = model_response
            response_b = baseline_response
            model_display_side = "A"
            baseline_display_side = "B"
        else:
            response_a = baseline_response
            response_b = model_response
            model_display_side = "B"
            baseline_display_side = "A"

        evaluation_id = f"AFV-{evaluation_number:03d}"
        evaluation_number += 1

        review_rows.append({
            "Evaluation_ID": evaluation_id,
            "Prompt": displayed_prompt,
            "Response_A": response_a,
            "Response_B": response_b,
            "Reviewer_Choice": "",
            "Comments": "",
        })

        llm_normalized = normalize_llm_result(
            score_row.get("model_pref")
        )

        if llm_normalized == "MODEL":
            llm_display_choice = model_display_side
        elif llm_normalized == "BASELINE":
            llm_display_choice = baseline_display_side
        elif llm_normalized == "TIE":
            llm_display_choice = "Tie"
        else:
            llm_display_choice = "Unknown"

        answer_key_rows.append({
            "Evaluation_ID": evaluation_id,
            "Matched_Unit_ID": unit["matched_unit_id"],
            "Base_ID": unit["base_id"],
            "Model": unit["model"],
            "Language": unit["language"],
            "Condition": CONDITION_LABELS[condition_key],
            "Typo_Rate": "0.4",
            "Source_File": source_file,
            "Original_Sample_ID": output_row.get("sample_id"),
            "Prompt": displayed_prompt,
            "Model_Response": model_response,
            "Baseline_Response": baseline_response,
            "Baseline_Field": baseline_field,
            "Displayed_Model_Side": model_display_side,
            "Displayed_Baseline_Side": baseline_display_side,
            "Original_LLM_Preference_AB": score_row.get("preference"),
            "Original_Model_Is_A": score_row.get("model_is_a"),
            "Original_Model_Pref": score_row.get("model_pref"),
            "LLM_Normalized_Result": llm_normalized,
            "LLM_Display_Choice": llm_display_choice,
            "LLM_Confidence": score_row.get("confidence"),
        })


# ============================================================
# Shuffle row order
# ============================================================

order = list(range(len(review_rows)))
display_rng.shuffle(order)

review_rows = [review_rows[i] for i in order]
answer_key_rows = [answer_key_rows[i] for i in order]

reviewer_df = pd.DataFrame(review_rows)
answer_key_df = pd.DataFrame(answer_key_rows)


# ============================================================
# Validate final files
# ============================================================

print("Reviewer rows:", len(reviewer_df))
print("Answer-key rows:", len(answer_key_df))
print(
    "Unique Evaluation IDs:",
    reviewer_df["Evaluation_ID"].nunique()
)

print("\nCondition counts:")
print(
    answer_key_df["Condition"]
    .value_counts()
    .sort_index()
)

print("\nLanguage counts:")
print(
    answer_key_df["Language"]
    .value_counts()
    .sort_index()
)

print("\nDisplayed model side counts:")
print(
    answer_key_df["Displayed_Model_Side"]
    .value_counts()
)

assert len(reviewer_df) == 96
assert len(answer_key_df) == 96
assert reviewer_df["Evaluation_ID"].nunique() == 96

expected_condition_counts = {
    "Clean": 24,
    "ESL-only": 24,
    "Typo-only": 24,
    "Combined": 24,
}

actual_condition_counts = (
    answer_key_df["Condition"].value_counts().to_dict()
)

assert actual_condition_counts == expected_condition_counts, (
    f"Unexpected condition counts: {actual_condition_counts}"
)


# ============================================================
# Export CSV files
# ============================================================

reviewer_1_path = (
    EXPORT_DIR
    / "alpaca_validation_reviewer_1.csv"
)

reviewer_2_path = (
    EXPORT_DIR
    / "alpaca_validation_reviewer_2.csv"
)

answer_key_path = (
    EXPORT_DIR
    / "alpaca_validation_private_answer_key.csv"
)

reviewer_df.to_csv(
    reviewer_1_path,
    index=False,
    encoding="utf-8-sig"
)

reviewer_df.to_csv(
    reviewer_2_path,
    index=False,
    encoding="utf-8-sig"
)

answer_key_df.to_csv(
    answer_key_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nFiles saved successfully:")
print(reviewer_1_path)
print(reviewer_2_path)
print(answer_key_path)

OSError: [Errno 30] Read-only file system: '/content/drive/MyDrive/openend2/alpaca_llm_judge_validation'

In [43]:
# ============================================================
# AlpacaFarm LLM Judge Validation
# Generate 96 human-review comparisons
#
# Requirements:
# - Google Drive is already mounted
# - sampled_units already exists
# - sampled_units contains 24 matched units
# - each unit contains:
#     clean
#     esl_only
#     typo_only
#     combined
# ============================================================

import json
import random
from pathlib import Path

import pandas as pd


# ============================================================
# 1. Paths and settings
# ============================================================

# Shared/read-only project folder
ROOT = Path("/content/drive/MyDrive/openend2")

# Original AlpacaFarm input files
INPUT_ROOT = ROOT / "alpaca"

# Save results to your own writable My Drive folder
EXPORT_DIR = Path(
    "/content/drive/MyDrive/alpaca_llm_judge_validation"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Test whether the output folder is writable
write_test_path = EXPORT_DIR / "_write_test.txt"

try:
    write_test_path.write_text(
        "write test",
        encoding="utf-8"
    )
    write_test_path.unlink()

    print(
        "Export folder is writable:",
        EXPORT_DIR
    )

except Exception as error:
    raise OSError(
        f"Cannot write to export folder:\n"
        f"{EXPORT_DIR}\n\n"
        f"Original error:\n{error}"
    )


RANDOM_SEED = 42

# Use a separate fixed seed for A/B placement and row order
display_rng = random.Random(
    RANDOM_SEED + 1000
)

TYPO_RATE = "0.4"

CONDITION_KEYS = [
    "clean",
    "esl_only",
    "typo_only",
    "combined",
]

CONDITION_LABELS = {
    "clean": "Clean",
    "esl_only": "ESL-only",
    "typo_only": "Typo-only",
    "combined": "Combined",
}

# According to run_alpaca_farm.py, any of these fields
# may contain the baseline/reference response.
BASELINE_KEYS = [
    "reference_output",
    "baseline_output",
    "output_ref",
    "chosen",
    "reference",
    "output",
]


# ============================================================
# 2. Helper functions
# ============================================================

def read_jsonl(path: Path) -> list[dict]:
    """
    Read a JSONL file safely.
    """

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(
                    json.loads(line)
                )

            except json.JSONDecodeError:
                print(
                    f"Warning: invalid JSON in "
                    f"{path}, line {line_number}"
                )

    return rows


# Cache source files so that each JSONL file
# is loaded only once.
input_file_cache = {}


def load_source_rows(
    source_file: str
) -> list[dict]:
    """
    Load the original AlpacaFarm input file.

    Example source_file values:

    raw_inputs/test.jsonl
    raw_inputs/arabic_esl_paper.jsonl
    typo_transforms/rate_0.4/test.jsonl
    typo_transforms/rate_0.4/arabic_esl_paper.jsonl
    """

    source_file = (
        str(source_file)
        .replace("\\", "/")
        .strip()
    )

    source_path = INPUT_ROOT / source_file

    if source_path not in input_file_cache:

        if not source_path.exists():
            raise FileNotFoundError(
                "Input source file not found:\n"
                f"{source_path}"
            )

        input_file_cache[source_path] = (
            read_jsonl(source_path)
        )

    return input_file_cache[source_path]


def get_base_row_index(
    base_id: str
) -> int:
    """
    Convert:

    row-688 -> 688
    """

    text = str(base_id).strip()

    if not text.startswith("row-"):
        raise ValueError(
            f"Unexpected base_id: {base_id}"
        )

    try:
        return int(
            text.split("-")[1]
        )

    except Exception as error:
        raise ValueError(
            f"Could not parse base_id: {base_id}"
        ) from error


def extract_baseline_response(
    source_row: dict
) -> tuple[str, str | None]:
    """
    Extract the baseline/reference response.

    In this AlpacaFarm dataset, the field 'output'
    contains the text-davinci-003 baseline response.
    """

    for key in BASELINE_KEYS:

        value = source_row.get(key)

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return value.strip(), key

    return "", None


def normalize_llm_result(
    model_pref
) -> str:
    """
    Definition from run_alpaca_farm.py:

    2 = evaluated model wins
    1 = baseline wins
    0 = tie
    """

    mapping = {
        2: "MODEL",
        1: "BASELINE",
        0: "TIE",
    }

    return mapping.get(
        model_pref,
        "UNKNOWN"
    )


# ============================================================
# 3. Basic checks on sampled_units
# ============================================================

if "sampled_units" not in globals():
    raise NameError(
        "sampled_units does not exist.\n"
        "Please run the matched sampling cell first."
    )

print(
    "Number of sampled matched units:",
    len(sampled_units)
)

if len(sampled_units) != 24:
    raise ValueError(
        "Expected 24 sampled matched units, "
        f"but found {len(sampled_units)}."
    )


# ============================================================
# 4. Build 96 pairwise evaluation rows
# ============================================================

review_rows = []
answer_key_rows = []

evaluation_number = 1

for unit in sampled_units:

    base_index = get_base_row_index(
        unit["base_id"]
    )

    for condition_key in CONDITION_KEYS:

        if condition_key not in unit:
            raise KeyError(
                f"Missing condition "
                f"{condition_key} in "
                f"{unit['matched_unit_id']}"
            )

        condition_data = unit[
            condition_key
        ]

        output_row = condition_data[
            "output_row"
        ]

        score_row = condition_data[
            "score_row"
        ]

        source_file = (
            output_row.get("source_file")
            or score_row.get("source_file")
        )

        if not source_file:
            raise ValueError(
                f"Missing source_file for "
                f"{unit['matched_unit_id']} / "
                f"{condition_key}"
            )

        source_rows = load_source_rows(
            source_file
        )

        if base_index >= len(source_rows):
            raise IndexError(
                f"Row index {base_index} is outside "
                f"{source_file}; file has "
                f"{len(source_rows)} rows."
            )

        source_row = source_rows[
            base_index
        ]

        baseline_response, baseline_field = (
            extract_baseline_response(
                source_row
            )
        )

        model_response = str(
            output_row.get(
                "output",
                ""
            )
        ).strip()

        instruction = str(
            output_row.get(
                "instruction",
                source_row.get(
                    "instruction",
                    ""
                )
            )
        ).strip()

        additional_input = str(
            output_row.get(
                "input",
                source_row.get(
                    "input",
                    ""
                )
            )
        ).strip()

        if additional_input:
            displayed_prompt = (
                f"{instruction}\n\n"
                f"Additional input:\n"
                f"{additional_input}"
            )
        else:
            displayed_prompt = instruction

        if not displayed_prompt:
            raise ValueError(
                f"Missing prompt for "
                f"{unit['matched_unit_id']} / "
                f"{condition_key}"
            )

        if not baseline_response:
            raise ValueError(
                f"Missing baseline response for "
                f"{unit['matched_unit_id']} / "
                f"{condition_key}"
            )

        if not model_response:
            raise ValueError(
                f"Missing model response for "
                f"{unit['matched_unit_id']} / "
                f"{condition_key}"
            )

        model_pref = score_row.get(
            "model_pref"
        )

        if model_pref not in [0, 1, 2]:
            raise ValueError(
                f"Invalid model_pref "
                f"for {unit['matched_unit_id']} / "
                f"{condition_key}: "
                f"{model_pref}"
            )

        # ----------------------------------------------------
        # Randomly decide whether the evaluated model
        # appears as Response A or Response B.
        #
        # This is independent of the original LLM judge order.
        # ----------------------------------------------------

        displayed_model_is_a = (
            display_rng.choice(
                [True, False]
            )
        )

        if displayed_model_is_a:

            response_a = model_response
            response_b = baseline_response

            model_display_side = "A"
            baseline_display_side = "B"

        else:

            response_a = baseline_response
            response_b = model_response

            model_display_side = "B"
            baseline_display_side = "A"

        evaluation_id = (
            f"AFV-{evaluation_number:03d}"
        )

        evaluation_number += 1

        # ----------------------------------------------------
        # Reviewer-visible row
        # ----------------------------------------------------

        review_rows.append({
            "Evaluation_ID": evaluation_id,
            "Prompt": displayed_prompt,
            "Response_A": response_a,
            "Response_B": response_b,
            "Reviewer_Choice": "",
            "Comments": "",
        })

        # ----------------------------------------------------
        # Private answer-key row
        # ----------------------------------------------------

        llm_normalized = (
            normalize_llm_result(
                model_pref
            )
        )

        if llm_normalized == "MODEL":

            llm_display_choice = (
                model_display_side
            )

        elif llm_normalized == "BASELINE":

            llm_display_choice = (
                baseline_display_side
            )

        elif llm_normalized == "TIE":

            llm_display_choice = "Tie"

        else:

            llm_display_choice = (
                "Unknown"
            )

        answer_key_rows.append({
            "Evaluation_ID": evaluation_id,
            "Matched_Unit_ID": (
                unit["matched_unit_id"]
            ),
            "Base_ID": unit["base_id"],
            "Model": unit["model"],
            "Language": unit["language"],
            "Condition": (
                CONDITION_LABELS[
                    condition_key
                ]
            ),
            "Typo_Rate": TYPO_RATE,
            "Source_File": source_file,
            "Original_Sample_ID": (
                output_row.get(
                    "sample_id"
                )
            ),
            "Prompt": displayed_prompt,
            "Response_A": response_a,
            "Response_B": response_b,
            "Model_Response": (
                model_response
            ),
            "Baseline_Response": (
                baseline_response
            ),
            "Baseline_Field": (
                baseline_field
            ),
            "Displayed_Model_Side": (
                model_display_side
            ),
            "Displayed_Baseline_Side": (
                baseline_display_side
            ),
            "Original_LLM_Preference_AB": (
                score_row.get(
                    "preference"
                )
            ),
            "Original_Model_Is_A": (
                score_row.get(
                    "model_is_a"
                )
            ),
            "Original_Model_Pref": (
                model_pref
            ),
            "LLM_Normalized_Result": (
                llm_normalized
            ),
            "LLM_Display_Choice": (
                llm_display_choice
            ),
            "LLM_Confidence": (
                score_row.get(
                    "confidence"
                )
            ),
        })


# ============================================================
# 5. Shuffle the 96 rows
# ============================================================

order = list(
    range(
        len(review_rows)
    )
)

display_rng.shuffle(order)

review_rows = [
    review_rows[index]
    for index in order
]

answer_key_rows = [
    answer_key_rows[index]
    for index in order
]

reviewer_df = pd.DataFrame(
    review_rows
)

answer_key_df = pd.DataFrame(
    answer_key_rows
)


# ============================================================
# 6. Validate final tables
# ============================================================

print(
    "\nReviewer rows:",
    len(reviewer_df)
)

print(
    "Answer-key rows:",
    len(answer_key_df)
)

print(
    "Unique Evaluation IDs:",
    reviewer_df[
        "Evaluation_ID"
    ].nunique()
)

print("\nCondition counts:")

print(
    answer_key_df[
        "Condition"
    ]
    .value_counts()
    .sort_index()
)

print("\nLanguage counts:")

print(
    answer_key_df[
        "Language"
    ]
    .value_counts()
    .sort_index()
)

print("\nDisplayed model side counts:")

print(
    answer_key_df[
        "Displayed_Model_Side"
    ]
    .value_counts()
)

print("\nLLM result counts:")

print(
    answer_key_df[
        "LLM_Normalized_Result"
    ]
    .value_counts()
)


# Required checks

assert len(reviewer_df) == 96, (
    f"Expected 96 reviewer rows, "
    f"found {len(reviewer_df)}."
)

assert len(answer_key_df) == 96, (
    f"Expected 96 answer-key rows, "
    f"found {len(answer_key_df)}."
)

assert (
    reviewer_df[
        "Evaluation_ID"
    ].nunique()
    == 96
), "Evaluation IDs are not unique."


expected_condition_counts = {
    "Clean": 24,
    "ESL-only": 24,
    "Typo-only": 24,
    "Combined": 24,
}

actual_condition_counts = (
    answer_key_df[
        "Condition"
    ]
    .value_counts()
    .to_dict()
)

assert (
    actual_condition_counts
    == expected_condition_counts
), (
    "Unexpected condition counts:\n"
    f"{actual_condition_counts}"
)


expected_language_counts = {
    language: 12
    for language in [
        "arabic",
        "french",
        "german",
        "japanese",
        "mandarin",
        "portuguese",
        "russian",
        "spanish",
    ]
}

actual_language_counts = (
    answer_key_df[
        "Language"
    ]
    .value_counts()
    .to_dict()
)

assert (
    actual_language_counts
    == expected_language_counts
), (
    "Unexpected language counts:\n"
    f"{actual_language_counts}"
)


# ============================================================
# 7. Export CSV files
# ============================================================

reviewer_1_path = (
    EXPORT_DIR
    / "alpaca_validation_reviewer_1.csv"
)

reviewer_2_path = (
    EXPORT_DIR
    / "alpaca_validation_reviewer_2.csv"
)

answer_key_path = (
    EXPORT_DIR
    / "alpaca_validation_private_answer_key.csv"
)


reviewer_df.to_csv(
    reviewer_1_path,
    index=False,
    encoding="utf-8-sig"
)

reviewer_df.to_csv(
    reviewer_2_path,
    index=False,
    encoding="utf-8-sig"
)

answer_key_df.to_csv(
    answer_key_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8. Final confirmation
# ============================================================

print(
    "\nFiles saved successfully:"
)

print(
    reviewer_1_path
)

print(
    reviewer_2_path
)

print(
    answer_key_path
)

print(
    "\nDo not send the private answer key "
    "to reviewers."
)

Export folder is writable: /content/drive/MyDrive/alpaca_llm_judge_validation
Number of sampled matched units: 24

Reviewer rows: 96
Answer-key rows: 96
Unique Evaluation IDs: 96

Condition counts:
Condition
Clean        24
Combined     24
ESL-only     24
Typo-only    24
Name: count, dtype: int64

Language counts:
Language
arabic        12
french        12
german        12
japanese      12
mandarin      12
portuguese    12
russian       12
spanish       12
Name: count, dtype: int64

Displayed model side counts:
Displayed_Model_Side
A    50
B    46
Name: count, dtype: int64

LLM result counts:
LLM_Normalized_Result
BASELINE    57
MODEL       29
TIE         10
Name: count, dtype: int64

Files saved successfully:
/content/drive/MyDrive/alpaca_llm_judge_validation/alpaca_validation_reviewer_1.csv
/content/drive/MyDrive/alpaca_llm_judge_validation/alpaca_validation_reviewer_2.csv
/content/drive/MyDrive/alpaca_llm_judge_validation/alpaca_validation_private_answer_key.csv

Do not send the pri

In [45]:
import pandas as pd
from pathlib import Path

EXPORT_DIR = Path(
    "/content/drive/MyDrive/alpaca_llm_judge_validation"
)

sampling_rows = []

for unit in sampled_units:
    sampling_rows.append({
        "Matched_Unit_ID": unit["matched_unit_id"],
        "Base_ID": unit["base_id"],
        "Language": unit["language"],
        "Model": unit["model"],
        "Random_Seed": 42,
        "Typo_Rate": 0.4,
    })

sampling_df = pd.DataFrame(sampling_rows)

sampling_path = (
    EXPORT_DIR
    / "alpaca_validation_sampled_units_seed42.csv"
)

sampling_df.to_csv(
    sampling_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully:")
print(sampling_path)

print("\nRows:", len(sampling_df))
display(sampling_df)

Saved successfully:
/content/drive/MyDrive/alpaca_llm_judge_validation/alpaca_validation_sampled_units_seed42.csv

Rows: 24


,Matched_Unit_ID,Base_ID,Language,Model,Random_Seed,Typo_Rate
0,Qwen2.5-7B-Instruct__arabic__row-688,row-688,arabic,Qwen2.5-7B-Instruct,42,0.4
1,Qwen2.5-7B-Instruct__arabic__row-200,row-200,arabic,Qwen2.5-7B-Instruct,42,0.4
2,Qwen2.5-7B-Instruct__arabic__row-120,row-120,arabic,Qwen2.5-7B-Instruct,42,0.4
3,Qwen2.5-7B-Instruct__french__row-782,row-782,french,Qwen2.5-7B-Instruct,42,0.4
4,Qwen2.5-7B-Instruct__french__row-351,row-351,french,Qwen2.5-7B-Instruct,42,0.4
5,Qwen2.5-7B-Instruct__french__row-323,row-323,french,Qwen2.5-7B-Instruct,42,0.4
6,Qwen2.5-7B-Instruct__german__row-303,row-303,german,Qwen2.5-7B-Instruct,42,0.4
7,Qwen2.5-7B-Instruct__german__row-226,row-226,german,Qwen2.5-7B-Instruct,42,0.4
8,Qwen2.5-7B-Instruct__german__row-778,row-778,german,Qwen2.5-7B-Instruct,42,0.4
9,Qwen2.5-7B-Instruct__japanese__row-192,row-192,japanese,Qwen2.5-7B-Instruct,42,0.4
